# DeepChessIQ NLP Module - Complete Demo

This comprehensive notebook demonstrates **ALL** features of the DeepChessIQ NLP module:

- ✅ Stockfish engine integration and testing
- ✅ PGN parsing and game analysis  
- ✅ Position evaluation and best move analysis
- ✅ Tactical motif detection (pins, forks, skewers, etc.)
- ✅ Feature extraction (material, development, center control)
- ✅ Turning point identification
- ✅ LLM-powered commentary generation
- ✅ Hallucination filtering
- ✅ Complete game processing with commentary
- ✅ Integration examples for RL team

**Requirements:**
- Python 3.8+
- All dependencies installed: `pip install -r requirements.txt`
- Stockfish executable available
- HuggingFace model access (for LLM features)



## 1. Setup and Configuration


In [1]:
# Install dependencies (uncomment if needed)
%pip install -r requirements.txt q

import sys
from pathlib import Path
import json

# Add module paths
current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.insert(0, str(current_dir))
sys.path.insert(0, str(parent_dir))

print(f"Current directory: {current_dir}")
print(f"Parent directory: {parent_dir}")


Note: you may need to restart the kernel to use updated packages.
Current directory: C:\Users\Admin\Desktop\AIP_DeepChessIQ\Nlp_final\deepchessiq_nlp
Parent directory: C:\Users\Admin\Desktop\AIP_DeepChessIQ\Nlp_final


In [2]:
# Import all required modules
import sys
from pathlib import Path

# Ensure chess module is available
try:
    import chess
    print("✓ python-chess module available")
except ImportError:
    print("✗ python-chess not installed. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-chess"])
    import chess
    print("✓ python-chess installed")

try:
    from deepchessiq_nlp.run_commentary import generate_commentary, CommentaryPipeline
    from deepchessiq_nlp.models.schemas import LLMConfig
    from deepchessiq_nlp.analysis.pgn_parser import parse_pgn, extract_positions, get_game_result
    from deepchessiq_nlp.analysis.feature_extractor import FeatureExtractor
    from deepchessiq_nlp.analysis.turning_point_detector import TurningPointDetector
    from deepchessiq_nlp.engine.tactical_motif_detector import TacticalMotifDetector
    from deepchessiq_nlp.engine.eval_classifier import EvalClassifier
    from deepchessiq_nlp.engine.python_chess_evaluator import PythonChessEvaluator
    from deepchessiq_nlp.utils.logger import logger
    
    # Try to import Stockfish (optional)
    try:
        from deepchessiq_nlp.engine.stockfish_wrapper import StockfishWrapper, STOCKFISH_PATH
        HAS_STOCKFISH = True
    except ImportError:
        HAS_STOCKFISH = False
        STOCKFISH_PATH = None
    
    print("✓ Imports successful (package mode)")
except ImportError:
    from run_commentary import generate_commentary, CommentaryPipeline
    from models.schemas import LLMConfig
    from analysis.pgn_parser import parse_pgn, extract_positions, get_game_result
    from analysis.feature_extractor import FeatureExtractor
    from analysis.turning_point_detector import TurningPointDetector
    from engine.tactical_motif_detector import TacticalMotifDetector
    from engine.eval_classifier import EvalClassifier
    from engine.python_chess_evaluator import PythonChessEvaluator
    from utils.logger import logger
    
    # Try to import Stockfish (optional)
    try:
        from engine.stockfish_wrapper import StockfishWrapper, STOCKFISH_PATH
        HAS_STOCKFISH = True
    except ImportError:
        HAS_STOCKFISH = False
        STOCKFISH_PATH = None
    
    print("✓ Imports successful (direct mode)")

print(f"\nStockfish available: {HAS_STOCKFISH}")
if HAS_STOCKFISH:
    print(f"Stockfish path: {STOCKFISH_PATH}")


✓ python-chess module available


C:\Users\Admin\anaconda3\envs\deepgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful (package mode)

Stockfish available: True
Stockfish path: C:\\Users\\Admin\\Desktop\\AIP_DeepChessIQ\\Nlp_final\\stockfish\\stockfish-windows-x86-64-avx2.exe


In [3]:
# Check system resources
import torch
try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    print("psutil not available - skipping RAM check")

print("System Resources:")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

if HAS_PSUTIL:
    memory = psutil.virtual_memory()
    print(f"\nRAM Total: {memory.total / 1e9:.2f} GB")
    print(f"RAM Available: {memory.available / 1e9:.2f} GB")
    print(f"RAM Used: {memory.used / 1e9:.2f} GB")

print("=" * 60)


System Resources:
PyTorch version: 2.5.1
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
GPU Memory: 8.59 GB

RAM Total: 16.34 GB
RAM Available: 3.01 GB
RAM Used: 13.33 GB


## 2. Position Evaluation Setup

**Note**: This module supports both Stockfish (if available) and PythonChessEvaluator (default).
If Stockfish is not available, the PythonChessEvaluator will be used automatically.


In [4]:
# Check evaluation engine availability
print("Evaluation Engine Configuration:")
print("=" * 60)

if HAS_STOCKFISH and STOCKFISH_PATH:
    stockfish_file = Path(STOCKFISH_PATH)
    print(f"Stockfish path: {STOCKFISH_PATH}")
    print(f"Exists: {stockfish_file.exists()}")
    
    if stockfish_file.exists():
        size_mb = stockfish_file.stat().st_size / 1024 / 1024
        print(f"File size: {size_mb:.2f} MB")
        print("✓ Stockfish executable found!")
        USE_STOCKFISH = True
    else:
        print("⚠ Stockfish path not found, will use PythonChessEvaluator")
        USE_STOCKFISH = False
else:
    print("⚠ Stockfish not available, will use PythonChessEvaluator")
    USE_STOCKFISH = False

print("\nFallback: PythonChessEvaluator (always available)")
print("=" * 60)


Evaluation Engine Configuration:
Stockfish path: C:\\Users\\Admin\\Desktop\\AIP_DeepChessIQ\\Nlp_final\\stockfish\\stockfish-windows-x86-64-avx2.exe
Exists: True
File size: 76.14 MB
✓ Stockfish executable found!

Fallback: PythonChessEvaluator (always available)


In [5]:
# Test position evaluation (Stockfish or PythonChessEvaluator)
print("Testing Position Evaluation:")
print("=" * 60)

try:
    if USE_STOCKFISH and HAS_STOCKFISH:
        print("Attempting to use Stockfish...")
        stockfish = StockfishWrapper()
        
        if stockfish.connect():
            print("✓ Connected to Stockfish successfully!")
            evaluator_name = "Stockfish"
            
            # Test evaluation
            test_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
            evaluation = stockfish.evaluate_position(test_fen, depth=10)
            
            print(f"\n✓ Evaluation successful!")
            print(f"  Score: {evaluation.score:.2f} pawns")
            print(f"  Best move: {evaluation.best_move}")
            print(f"  Depth: {evaluation.depth}")
            
            stockfish.disconnect()
        else:
            print("⚠ Stockfish connection failed, using PythonChessEvaluator")
            evaluator = PythonChessEvaluator()
            evaluator_name = "PythonChessEvaluator"
            test_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
            evaluation = evaluator.evaluate_position(test_fen)
            print(f"\n✓ Evaluation successful (PythonChessEvaluator)!")
            print(f"  Score: {evaluation.score:.2f} pawns")
            print(f"  Best move: {evaluation.best_move}")
    else:
        print("Using PythonChessEvaluator (no Stockfish)")
        evaluator = PythonChessEvaluator()
        evaluator_name = "PythonChessEvaluator"
        test_fen = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
        evaluation = evaluator.evaluate_position(test_fen)
        print(f"\n✓ Evaluation successful!")
        print(f"  Score: {evaluation.score:.2f} pawns")
        print(f"  Best move: {evaluation.best_move}")
        print(f"  (Note: PythonChessEvaluator uses heuristics, not Stockfish engine)")
    
    print(f"\n✓ Using {evaluator_name} for position evaluation")
    
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

print("=" * 60)


Testing Position Evaluation:
Attempting to use Stockfish...
2025-11-28 18:57:50,222 - deepchessiq_nlp - INFO - connect:82 - Attempting to connect to Stockfish at: C:\\Users\\Admin\\Desktop\\AIP_DeepChessIQ\\Nlp_final\\stockfish\\stockfish-windows-x86-64-avx2.exe
2025-11-28 18:57:50,226 - deepchessiq_nlp - ERROR - connect:116 - Failed to connect to Stockfish at: C:\\Users\\Admin\\Desktop\\AIP_DeepChessIQ\\Nlp_final\\stockfish\\stockfish-windows-x86-64-avx2.exe
Error type: NotImplementedError
Error message: 
Make sure the Stockfish executable is valid and executable.
ERROR: Failed to connect to Stockfish at: C:\\Users\\Admin\\Desktop\\AIP_DeepChessIQ\\Nlp_final\\stockfish\\stockfish-windows-x86-64-avx2.exe
Error type: NotImplementedError
Error message: 
Make sure the Stockfish executable is valid and executable.
⚠ Stockfish connection failed, using PythonChessEvaluator

✓ Evaluation successful (PythonChessEvaluator)!
  Score: 2.00 pawns
  Best move: Nh3

✓ Using PythonChessEvaluator for 

## 3. PGN Parsing and Game Analysis


In [6]:
# Parse a PGN file
pgn_path = "data/sample_game.pgn"

print("PGN Parsing:")
print("=" * 60)

game = parse_pgn(pgn_path)

if game:
    print(f"✓ Successfully parsed PGN file")
    print(f"\nGame Information:")
    print(f"  Event: {game.headers.get('Event', 'Unknown')}")
    print(f"  White: {game.headers.get('White', 'Unknown')}")
    print(f"  Black: {game.headers.get('Black', 'Unknown')}")
    print(f"  Date: {game.headers.get('Date', 'Unknown')}")
    print(f"  Result: {game.headers.get('Result', 'Unknown')}")
    
    # Extract positions
    positions = extract_positions(game)
    print(f"\n  Total moves: {len(positions)}")
    print(f"\n  First 5 moves:")
    for pos in positions[:5]:
        print(f"    Move {pos['move_number']}: {pos['move_san']}")
else:
    print("✗ Failed to parse PGN file")

print("=" * 60)


PGN Parsing:
2025-11-28 18:57:53,023 - deepchessiq_nlp - INFO - parse_pgn:35 - Successfully parsed PGN: Sample Game
✓ Successfully parsed PGN file

Game Information:
  Event: Sample Game
  White: Player A
  Black: Player B
  Date: 2024.01.01
  Result: 1-0

  Total moves: 85

  First 5 moves:
    Move 1: e4
    Move 2: e5
    Move 3: Nf3
    Move 4: Nc6
    Move 5: Bb5


## 4. Position Evaluation and Analysis


In [7]:
# Evaluate positions from the game
print("Position Evaluation:")
print("=" * 60)

if 'game' in locals() and game:
    positions = extract_positions(game)
    
    # Use PythonChessEvaluator (always available)
    evaluator = PythonChessEvaluator()
    
    print("\nEvaluating first 5 positions...\n")
    
    for i, pos in enumerate(positions[:5]):
        print(f"Move {pos['move_number']}: {pos['move_san']}")
        evaluation = evaluator.evaluate_position(pos['position_fen'])
        
        # Classify evaluation
        is_white_to_move = pos['is_white']
        classification = EvalClassifier.classify_evaluation(evaluation, is_white_to_move)
        
        print(f"  Evaluation: {evaluation.score:+.2f} pawns ({classification['description']})")
        print(f"  Best move: {evaluation.best_move}")
        print()

print("=" * 60)


Position Evaluation:

Evaluating first 5 positions...

Move 1: e4
  Evaluation: -2.00 pawns (Losing position)
  Best move: Nh6

Move 2: e5
  Evaluation: +2.90 pawns (Losing position)
  Best move: Nh3

Move 3: Nf3
  Evaluation: -2.90 pawns (Losing position)
  Best move: Ne7

Move 4: Nc6
  Evaluation: +2.70 pawns (Losing position)
  Best move: Ng5

Move 5: Bb5
  Evaluation: -3.00 pawns (Losing position)
  Best move: Nge7



## 5. Tactical Motif Detection


In [8]:
# Detect tactical motifs
print("Tactical Motif Detection:")
print("=" * 60)

if 'game' in locals() and game:
    positions = extract_positions(game)
    motif_detector = TacticalMotifDetector()
    
    print("\nAnalyzing positions for tactical motifs...\n")
    
    for pos in positions[:10]:
        board = pos['board']
        motifs = motif_detector.detect_motifs(board)
        
        if motifs:
            print(f"Move {pos['move_number']}: {pos['move_san']}")
            for motif in motifs:
                print(f"  • {motif.name}: {motif.description}")
            print()

print("=" * 60)


Tactical Motif Detection:

Analyzing positions for tactical motifs...

Move 1: e4
  • fork: Fork opportunity with Q
  • castling_opportunity: Kingside castling available for white
  • castling_opportunity: Queenside castling available for white
  • castling_opportunity: Kingside castling available for black
  • castling_opportunity: Queenside castling available for black
  • undeveloped_piece: Undeveloped knight on b1
  • undeveloped_piece: Undeveloped knight on g1
  • undeveloped_piece: Undeveloped knight on b8
  • undeveloped_piece: Undeveloped knight on g8

Move 2: e5
  • fork: Fork opportunity with Q
  • castling_opportunity: Kingside castling available for white
  • castling_opportunity: Queenside castling available for white
  • castling_opportunity: Kingside castling available for black
  • castling_opportunity: Queenside castling available for black
  • undeveloped_piece: Undeveloped knight on b1
  • undeveloped_piece: Undeveloped knight on g1
  • undeveloped_piece: Undeveloped

## 6. Feature Extraction


In [9]:
# Extract position features
print("Feature Extraction:")
print("=" * 60)

if 'game' in locals() and game:
    positions = extract_positions(game)
    feature_extractor = FeatureExtractor()
    
    print("\nExtracting features from positions...\n")
    
    for pos in positions[:3]:
        board = pos['board']
        features = feature_extractor.extract_position_features(board)
        
        print(f"Move {pos['move_number']}: {pos['move_san']}")
        print(f"  Material balance: {features['material_balance']:+.1f}")
        print(f"  Center control: {features['center_control']['control_balance']:+d}")
        print(f"  Development: {features['development']['development_balance']:+d} pieces")
        print()

print("=" * 60)


Feature Extraction:

Extracting features from positions...

Move 1: e4
  Material balance: +0.0
  Center control: +1
  Development: +0 pieces

Move 2: e5
  Material balance: +0.0
  Center control: +0
  Development: +0 pieces

Move 3: Nf3
  Material balance: +0.0
  Center control: +2
  Development: +1 pieces



## 7. LLM Commentary Generation (Full Pipeline)

⚠️ **WARNING**: Loading the LLM model requires significant memory:
- **Phi-3-mini with 4-bit quantization: ~2-3GB RAM** (Recommended for your system)
- Full precision: ~8GB RAM
- Loading takes 2-5 minutes

**Default Model**: `microsoft/Phi-3-mini-4k-instruct` (open model, no authentication required)


### ✅ Using Phi-3-mini (Open Model, No Authentication Required)

**Default Model**: `microsoft/Phi-3-mini-4k-instruct`
- ✅ **No authentication required** (open model)
- ✅ **Memory efficient** (~2-3GB with 4-bit quantization)
- ✅ **Optimized for your system** (16GB RAM, 8.59GB GPU)

**Note**: If you prefer to use Gemma-2-2B-IT (gated model), you'll need HuggingFace authentication. See alternative configuration in code below.


In [10]:
# Enable detailed download progress tracking
import sys
from huggingface_hub import HfApi
from tqdm.auto import tqdm
import os

def show_download_progress(model_name):
    """Show download progress for model files."""
    print("="*60)
    print("📥 MODEL DOWNLOAD TRACKER")
    print("="*60)
    print(f"Model: {model_name}")
    print("\nChecking model files...")
    
    try:
        api = HfApi()
        model_info = api.model_info(model_name)
        
        total_size = 0
        files_to_download = []
        
        # Get file sizes
        for file in model_info.siblings:
            if file.rfilename.endswith(('.safetensors', '.bin', '.json', '.txt', '.py')):
                size_mb = file.size / (1024 * 1024) if file.size else 0
                total_size += file.size if file.size else 0
                files_to_download.append((file.rfilename, size_mb))
        
        total_size_gb = total_size / (1024 * 1024 * 1024)
        
        print(f"\n📦 Total model size: {total_size_gb:.2f} GB")
        print(f"📁 Files to download: {len(files_to_download)}")
        
        print("\n🔍 Major files:")
        for filename, size_mb in files_to_download[:10]:  # Show first 10 files
            if size_mb > 10:  # Only show files larger than 10MB
                print(f"   • {filename}: {size_mb:.1f} MB")
        
        if len(files_to_download) > 10:
            print(f"   ... and {len(files_to_download) - 10} more files")
        
        print(f"\n⏱️  Estimated download time:")
        print(f"   • Fast connection (50+ Mbps): ~2-5 minutes")
        print(f"   • Medium connection (10-50 Mbps): ~5-10 minutes")
        print(f"   • Slow connection (<10 Mbps): ~10-20 minutes")
        
        print(f"\n💡 Progress bars will appear automatically during download")
        print(f"💡 First download takes longer - subsequent loads use cache (<1 min)")
        print("="*60)
        
    except Exception as e:
        print(f"⚠ Could not get model info: {e}")
        print("Download will still work - progress bars will show during download")
        print("="*60)

# Check download status
print("Checking download requirements for Phi-3-mini...")
show_download_progress("microsoft/Phi-3-mini-4k-instruct")


Checking download requirements for Phi-3-mini...
📥 MODEL DOWNLOAD TRACKER
Model: microsoft/Phi-3-mini-4k-instruct

Checking model files...

📦 Total model size: 0.00 GB
📁 Files to download: 12

🔍 Major files:
   ... and 2 more files

⏱️  Estimated download time:
   • Fast connection (50+ Mbps): ~2-5 minutes
   • Medium connection (10-50 Mbps): ~5-10 minutes
   • Slow connection (<10 Mbps): ~10-20 minutes

💡 Progress bars will appear automatically during download
💡 First download takes longer - subsequent loads use cache (<1 min)


In [12]:
# HuggingFace Authentication Setup (OPTIONAL)
# Only needed if you want to use Gemma-2-2B-IT instead of Phi-3
# Phi-3 is open and doesn't require authentication

import os
from huggingface_hub import login, whoami
from getpass import getpass

print("HuggingFace Authentication Check:")
print("=" * 60)

# Check if already logged in
try:
    user_info = whoami()
    print(f"✓ Already authenticated as: {user_info.get('name', 'Unknown')}")
    HF_TOKEN = None  # Will use cached token
    print("✓ Using cached HuggingFace token")
except Exception as e:
    print("⚠ Not authenticated. You need to authenticate to access gated models.")
    print("\nOption 1: Login via CLI (recommended)")
    print("  Run in terminal: huggingface-cli login")
    print("\nOption 2: Set token manually")
    print("  Get token from: https://huggingface.co/settings/tokens")
    
    # Option to set token manually
    use_manual_token = input("\nDo you want to set a token now? (y/n): ").strip().lower()
    
    if use_manual_token == 'y':
        token = getpass("Enter your HuggingFace token: ").strip()
        if token:
            try:
                login(token=token)
                HF_TOKEN = token
                print("✓ Authentication successful!")
            except Exception as e:
                print(f"✗ Authentication failed: {e}")
                HF_TOKEN = None
        else:
            print("⚠ No token provided")
            HF_TOKEN = None
    else:
        print("⚠ Skipping token setup. Model loading may fail if not authenticated.")
        HF_TOKEN = None

# Check model access (optional - only if authenticated)
if HF_TOKEN or 'user_info' in locals():
    try:
        from huggingface_hub import model_info
        model_info("google/gemma-2-2b-it", token=HF_TOKEN)
        print("\n✓ Model access verified!")
    except Exception as e:
        if "401" in str(e) or "gated" in str(e).lower():
            print(f"\n⚠ WARNING: Cannot access gated model: {e}")
            print("  Please request access at: https://huggingface.co/google/gemma-2-2b-it")
        else:
            print(f"\n⚠ Could not verify model access: {e}")

print("=" * 60)
if HF_TOKEN or 'user_info' in locals():
    print("\n✓ Authentication ready. You can use Gemma-2-2B-IT if desired.")
else:
    print("\n⚠ No authentication needed for Phi-3 (default model).")
    print("   If you want to use Gemma, authenticate first and then change the model in the config cell below.")


HuggingFace Authentication Check:
⚠ Not authenticated. You need to authenticate to access gated models.

Option 1: Login via CLI (recommended)
  Run in terminal: huggingface-cli login

Option 2: Set token manually
  Get token from: https://huggingface.co/settings/tokens



Do you want to set a token now? (y/n):  y
Enter your HuggingFace token:  ········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Authentication successful!

✓ Model access verified!

✓ Authentication ready. You can use Gemma-2-2B-IT if desired.


✓ Authentication successful!

✓ Model access verified!

✓ Authentication ready. You can use Gemma-2-2B-IT if desired.


✓ Authentication successful!

✓ Model access verified!

✓ Authentication ready. You can use Gemma-2-2B-IT if desired.


✓ Authentication successful!

✓ Model access verified!

✓ Authentication ready. You can use Gemma-2-2B-IT if desired.


✓ Authentication successful!

✓ Model access verified!

✓ Authentication ready. You can use Gemma-2-2B-IT if desired.


✓ Authentication successful!

✓ Model access verified!

Next: Configure LLM with token in the cell below.


✓ Authentication successful!

✓ Model access verified!

Next: Configure LLM with token in the cell below.


In [13]:
# Configure LLM - Using Phi-3-mini by default (open model, no auth required)
# Phi-3-mini is optimized for systems with limited RAM and doesn't require authentication

# Reload schemas module to get latest LLMConfig
import importlib
try:
    from deepchessiq_nlp.models import schemas
    importlib.reload(schemas)
    from deepchessiq_nlp.models.schemas import LLMConfig
    print("✓ Reloaded schemas module to get latest LLMConfig")
except ImportError:
    try:
        import schemas
        importlib.reload(schemas)
        from models.schemas import LLMConfig
        print("✓ Reloaded schemas module to get latest LLMConfig")
    except Exception as e:
        print(f"⚠ Could not reload schemas module: {e}")
        print("   If you see errors, restart the kernel and re-run all cells")

# Check if bitsandbytes is available for quantization
print("\nChecking quantization support...")
try:
    import bitsandbytes
    HAS_QUANTIZATION = True
    print("✓ bitsandbytes available - quantization supported")
except ImportError:
    HAS_QUANTIZATION = False
    print("⚠ bitsandbytes not available - will use full precision")
    print("  Install with: pip install bitsandbytes")
    print("  Note: bitsandbytes may have issues on Windows")

# Configure based on quantization availability
if HAS_QUANTIZATION:
    # Option 1: Phi-3-mini with 4-bit quantization (RECOMMENDED if bitsandbytes available)
    # - No authentication required
    # - ~2-3GB RAM usage
    # - Optimized for systems with limited RAM
    llm_config = LLMConfig(
        model_name="microsoft/Phi-3-mini-4k-instruct",  # Open model, no auth needed
        temperature=0.7,
        max_new_tokens=250,  # Increased to ensure 2+ sentences
        min_new_tokens=50,   # Minimum for 2 sentences
        use_gpu=True,
        load_in_4bit=True,  # 4-bit quantization for memory efficiency
        hf_token=None  # Not needed for Phi-3 (open model)
    )
    print("\n✓ Using Phi-3-mini with 4-bit quantization")
else:
    # Option 2: Phi-3-mini with full precision (fallback if bitsandbytes unavailable)
    # - No authentication required
    # - ~6-8GB RAM usage (still manageable)
    llm_config = LLMConfig(
        model_name="microsoft/Phi-3-mini-4k-instruct",  # Open model, no auth needed
        temperature=0.7,
        max_new_tokens=250,  # Increased to ensure 2+ sentences
        min_new_tokens=50,   # Minimum for 2 sentences
        use_gpu=True,
        load_in_4bit=False,  # Full precision (quantization not available)
        hf_token=None  # Not needed for Phi-3 (open model)
    )
    print("\n✓ Using Phi-3-mini with full precision (quantization not available)")

# Alternative Option 2: Phi-3-mini without quantization (if you have more RAM)
# llm_config = LLMConfig(
#     model_name="microsoft/Phi-3-mini-4k-instruct",
#     temperature=0.7,
#     max_new_tokens=250,
#     min_new_tokens=50,
#     use_gpu=True,
#     load_in_4bit=False  # Full precision (~8GB RAM)
# )

# Alternative Option 3: Gemma-2-2B-IT (requires HuggingFace authentication)
# Uncomment below and run authentication cell first:
# llm_config = LLMConfig(
#     model_name="google/gemma-2-2b-it",
#     temperature=0.7,
#     max_new_tokens=250,
#     min_new_tokens=50,
#     use_gpu=True,
#     load_in_4bit=True,
#     hf_token=hf_token  # Requires HuggingFace authentication
# )

print(f"\n{'='*60}")
print("LLM Configuration:")
print(f"{'='*60}")
print(f"  Model: {llm_config.model_name}")
print(f"  4-bit quantization: {llm_config.load_in_4bit}")
print(f"  Use GPU: {llm_config.use_gpu}")
print(f"  Temperature: {llm_config.temperature}")
print(f"  Max tokens: {llm_config.max_new_tokens}")
print(f"  Min tokens: {getattr(llm_config, 'min_new_tokens', 'N/A')}")

# Check model type and authentication requirements
if "phi-3" in llm_config.model_name.lower():
    print(f"\n✓ Phi-3-mini configured (open model, no authentication required)")
    print(f"✓ Optimized for systems with limited RAM (~2-3GB with 4-bit quantization)")
elif "gemma" in llm_config.model_name.lower():
    has_token = False
    if hasattr(llm_config, 'hf_token'):
        token_value = getattr(llm_config, 'hf_token', None)
        has_token = token_value is not None and token_value != ""
    print(f"  HuggingFace token: {'✓ Set' if has_token else '✗ Not set (may fail for gated models)'}")
    if not has_token:
        print("\n⚠️  WARNING: Gemma requires HuggingFace authentication.")
        print("   Run the authentication cell above or use: huggingface-cli login")

print(f"\n✓ Model configured to generate at least 2 sentences")
print(f"{'='*60}")


✓ Reloaded schemas module to get latest LLMConfig

Checking quantization support...
✓ bitsandbytes available - quantization supported

✓ Using Phi-3-mini with 4-bit quantization

LLM Configuration:
  Model: microsoft/Phi-3-mini-4k-instruct
  4-bit quantization: True
  Use GPU: True
  Temperature: 0.7
  Max tokens: 250
  Min tokens: 50

✓ Phi-3-mini configured (open model, no authentication required)
✓ Optimized for systems with limited RAM (~2-3GB with 4-bit quantization)

✓ Model configured to generate at least 2 sentences


In [14]:
# Initialize pipeline with LLM
print("Initializing pipeline (this may take several minutes)...")
print("=" * 60)

try:
    pipeline = CommentaryPipeline(llm_config=llm_config)
    print("✓ Pipeline initialized successfully!")
except MemoryError:
    print("✗ ERROR: Out of memory!")
    print("   Try using 4-bit quantization (set load_in_4bit=True)")
    raise
except Exception as e:
    print(f"✗ ERROR: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    raise

print("=" * 60)


Initializing pipeline (this may take several minutes)...
2025-11-28 18:59:47,060 - deepchessiq_nlp - INFO - _detect_device:39 - Using GPU: NVIDIA GeForce RTX 4070 Laptop GPU
2025-11-28 18:59:47,060 - deepchessiq_nlp - INFO - __init__:78 - ✓ CommentaryPipeline initialized
✓ Pipeline initialized successfully!


In [15]:
# Generate commentary for a single move
print("Generating Commentary for Single Move:")
print("=" * 60)

try:
    # Example position after 1.e4 e5 2.Nf3
    position_fen = "rnbqkbnr/pppp1ppp/8/4p3/4P3/5N2/PPPP1PPP/RNBQKB1R w KQkq - 0 1"
    move_san = "Nf3"
    
    print(f"Position: After 1.e4 e5 2.Nf3")
    print(f"Move: {move_san}\n")
    
    result = pipeline.generate_single_move_commentary(
        position_fen=position_fen,
        move_san=move_san,
        move_number=3,
        previous_moves=["e4", "e5"]
    )
    
    print(f"Evaluation: {result.move_analysis.evaluation.score:+.2f} pawns")
    print(f"Best move: {result.move_analysis.evaluation.best_move}")
    print(f"\nCommentary:")
    print(f"{result.commentary}")
    print(f"\nHallucination check passed: {result.hallucination_check_passed}")
    
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

print("=" * 60)


Generating Commentary for Single Move:
Position: After 1.e4 e5 2.Nf3
Move: Nf3

2025-11-28 18:59:48,359 - deepchessiq_nlp - INFO - load_model:52 - Loading model: microsoft/Phi-3-mini-4k-instruct
2025-11-28 18:59:48,360 - deepchessiq_nlp - INFO - load_model:53 - This may take 2-5 minutes and use significant memory...

📥 DOWNLOAD PROGRESS
Model: microsoft/Phi-3-mini-4k-instruct
First-time download may take 5-15 minutes (normal)
Progress bars will appear below as files download...

2025-11-28 18:59:48,361 - deepchessiq_nlp - INFO - load_model:81 - Using HuggingFace token for authentication

📥 Downloading tokenizer files...
✓ Tokenizer downloaded and loaded

2025-11-28 18:59:48,867 - deepchessiq_nlp - INFO - load_model:133 - Using 4-bit quantization

📥 DOWNLOADING MODEL WEIGHTS
This is the large file (~7-8GB) - may take several minutes
Progress bars will show below as files download...



`torch_dtype` is deprecated! Use `dtype` instead!
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████| 2/2 [00:16<00:00,  8.49s/it]



✓ MODEL WEIGHTS DOWNLOADED AND LOADED

2025-11-28 19:00:12,598 - deepchessiq_nlp - INFO - load_model:189 - Successfully loaded model: microsoft/Phi-3-mini-4k-instruct


You are not running the flash-attention implementation, expect numerical differences.


2025-11-28 19:00:38,829 - deepchessiq_nlp - INFO - generate_commentary:87 - Generated commentary with 5 sentences
Evaluation: +2.70 pawns
Best move: Ng5

Commentary:
The knight'issance of White with 3. Nf3 solidifies control over d4 while supporting potential central advancement after Black responds. This aggressive yet flexible development creates immediate threats without compromising position flexibility or king safety. With engines favoring an imminent fork from Ng5 against f7 – often dubbed as "the weakest point" due to its tendency being guarded by just one defender post-castling – it remains crucial that players consider such tactical opportunities early on. Notably absent is any direct threat towards castling kingside; however, if left unchecked, White can exploit these open lines later, especially once they maneuver their pieces into stronger positions around the center.

Hallucination check passed: True


## 8. Process Complete Game with Commentary

⚠️ **Note**: Processing full games takes significant time. Limited to 5 moves for demo.


## 9. Commentary Quality Evaluation Metrics

This section evaluates the quality of generated commentary using various NLP metrics:
- **Sentence Count**: Ensures minimum 2 sentences requirement
- **Length Metrics**: Word count, character count
- **Readability**: Basic readability scores
- **Chess-Specific**: Checks for chess terminology usage
- **Quality Indicators**: Coherence and structure analysis


In [ ]:
# BLEU and ROUGE Score Evaluation Functions

def calculate_bleu_score(candidate, references, max_n=4):
    """
    Calculate BLEU score for generated commentary.
    
    Args:
        candidate: Generated commentary text (string)
        references: List of reference commentary texts (list of strings)
        max_n: Maximum n-gram order (default: 4)
    
    Returns:
        BLEU score (float, 0-1)
    """
    try:
        from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
        from nltk.tokenize import word_tokenize
        
        # Tokenize candidate
        candidate_tokens = word_tokenize(candidate.lower())
        
        # Tokenize references
        reference_tokens = [word_tokenize(ref.lower()) for ref in references]
        
        # Use smoothing to handle zero matches
        smoothing = SmoothingFunction().method1
        
        # Calculate BLEU score
        bleu_score = sentence_bleu(
            reference_tokens,
            candidate_tokens,
            smoothing_function=smoothing
        )
        
        return round(bleu_score, 4)
    
    except ImportError:
        print("⚠️  NLTK not available. Install with: pip install nltk")
        return None
    except Exception as e:
        print(f"⚠️  Error calculating BLEU: {e}")
        return None


def calculate_rouge_scores(candidate, reference):
    """
    Calculate ROUGE-1, ROUGE-2, and ROUGE-L scores.
    
    Args:
        candidate: Generated commentary text (string)
        reference: Reference commentary text (string)
    
    Returns:
        Dictionary with rouge_1, rouge_2, rouge_l scores
    """
    try:
        from rouge_score import rouge_scorer
        
        # Note: rouge-score library uses 'rougeL' with capital L
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        scores = scorer.score(reference, candidate)
        
        return {
            'rouge_1': round(scores['rouge1'].fmeasure, 4),
            'rouge_1_precision': round(scores['rouge1'].precision, 4),
            'rouge_1_recall': round(scores['rouge1'].recall, 4),
            'rouge_2': round(scores['rouge2'].fmeasure, 4),
            'rouge_2_precision': round(scores['rouge2'].precision, 4),
            'rouge_2_recall': round(scores['rouge2'].recall, 4),
            'rouge_l': round(scores['rougeL'].fmeasure, 4),
            'rouge_l_precision': round(scores['rougeL'].precision, 4),
            'rouge_l_recall': round(scores['rougeL'].recall, 4),
        }
    
    except ImportError:
        print("⚠️  rouge-score not available. Install with: pip install rouge-score")
        return None
    except Exception as e:
        print(f"⚠️  Error calculating ROUGE: {e}")
        return None


def evaluate_with_references(candidate_text, reference_texts):
    """
    Comprehensive evaluation with BLEU and ROUGE scores when references are available.
    
    Args:
        candidate_text: Generated commentary to evaluate
        reference_texts: List of reference commentaries (or single string)
    
    Returns:
        Dictionary with BLEU and ROUGE scores
    """
    results = {}
    
    # Handle single reference as list
    if isinstance(reference_texts, str):
        reference_texts = [reference_texts]
    
    # Calculate BLEU
    bleu = calculate_bleu_score(candidate_text, reference_texts)
    if bleu is not None:
        results['bleu_score'] = bleu
    
    # Calculate ROUGE (use first reference)
    if reference_texts:
        rouge = calculate_rouge_scores(candidate_text, reference_texts[0])
        if rouge:
            results.update(rouge)
    
    return results


# Example: Evaluate commentary with reference (if available)
print("="*60)
print("BLEU and ROUGE Score Evaluation")
print("="*60)
print("\n📝 Note: BLEU and ROUGE scores require reference commentaries for comparison.")
print("   These metrics are useful when evaluating against expert commentary datasets.")
print("\n✅ Functions available:")
print("   • calculate_bleu_score(candidate, references)")
print("   • calculate_rouge_scores(candidate, reference)")
print("   • evaluate_with_references(candidate, references)")
print("\n" + "="*60)


In [28]:
# Example: Using BLEU and ROUGE Scores with Generated Commentary

# Example reference commentaries (in real use, these would come from expert annotations)
example_references = [
    "White plays e4, establishing central control. This aggressive opening move immediately contests the center and prepares for rapid piece development.",
    "The move e4 opens the game for White, claiming the central square and allowing the bishop and queen to become active. Black must respond carefully to avoid falling into positional disadvantages.",
    "With e4, White adopts a classical approach, aiming for central dominance. This move enables quick development of the king's bishop and prepares for potential castling on the kingside."
]

# Evaluate generated commentary against references (if result exists)
if 'result' in locals():
    print("BLEU and ROUGE Evaluation Example:")
    print("=" * 60)
    
    # Get commentary from result (handle different formats)
    commentary_text = None
    
    # Try to extract commentary
    if isinstance(result, dict):
        if 'game_analysis' in result and 'moves' in result['game_analysis']:
            if result['game_analysis']['moves']:
                commentary_text = result['game_analysis']['moves'][0].get('commentary', '')
        elif 'commentary' in result:
            commentary_text = result.get('commentary', '')
    elif hasattr(result, 'commentary'):
        commentary_text = result.commentary
    elif hasattr(result, 'moves') and result.moves:
        commentary_text = result.moves[0].commentary
    
    if commentary_text:
        print(f"\n📝 Generated Commentary:")
        print(f"   {commentary_text[:200]}...")
        
        print(f"\n📊 Evaluating against {len(example_references)} reference commentaries...")
        
        # Calculate BLEU
        bleu = calculate_bleu_score(commentary_text, example_references)
        if bleu is not None:
            print(f"\n🎯 BLEU Score: {bleu:.4f}")
            print(f"   (Range: 0-1, higher is better)")
        
        # Calculate ROUGE
        rouge = calculate_rouge_scores(commentary_text, example_references[0])
        if rouge:
            print(f"\n📈 ROUGE Scores:")
            print(f"   • ROUGE-1 (F1): {rouge['rouge_1']:.4f} (P: {rouge['rouge_1_precision']:.4f}, R: {rouge['rouge_1_recall']:.4f})")
            print(f"   • ROUGE-2 (F1): {rouge['rouge_2']:.4f} (P: {rouge['rouge_2_precision']:.4f}, R: {rouge['rouge_2_recall']:.4f})")
            print(f"   • ROUGE-L (F1): {rouge['rouge_l']:.4f} (P: {rouge['rouge_l_precision']:.4f}, R: {rouge['rouge_l_recall']:.4f})")
            print(f"   (Precision = accuracy, Recall = coverage, F1 = balanced score)")
        
        # Comprehensive evaluation
        print(f"\n📋 Comprehensive Evaluation:")
        all_scores = evaluate_with_references(commentary_text, example_references)
        if all_scores:
            for metric, value in all_scores.items():
                if isinstance(value, float):
                    print(f"   • {metric}: {value:.4f}")
        
        print(f"\n💡 Tips:")
        print(f"   • Higher BLEU/ROUGE scores indicate better similarity to reference commentaries")
        print(f"   • ROUGE-1: Word overlap, ROUGE-2: Bigram overlap, ROUGE-L: Longest common subsequence")
        print(f"   • Use these metrics when comparing against expert-annotated commentary datasets")
        
    else:
        print("\n⚠️  No commentary found in result. Generate commentary first.")
else:
    print("⚠️  No result found. Generate commentary first, then run this cell.")
    
print("\n" + "=" * 60)


BLEU and ROUGE Evaluation Example:

📝 Generated Commentary:
   The opening with 1. e4 is highly aggressive, but Black'ieves an immediate developmental leap after responding appropriately; however, given White’s lack of coordination between pieces post-move, it se...

📊 Evaluating against 3 reference commentaries...

🎯 BLEU Score: 0.0052
   (Range: 0-1, higher is better)

📈 ROUGE Scores:
   • ROUGE-1 (F1): 0.1961 (P: 0.1128, R: 0.7500)
   • ROUGE-2 (F1): 0.0132 (P: 0.0076, R: 0.0526)
   • ROUGE-L (F1): 0.0784 (P: 0.0451, R: 0.3000)
   (Precision = accuracy, Recall = coverage, F1 = balanced score)

📋 Comprehensive Evaluation:
   • bleu_score: 0.0052
   • rouge_1: 0.1961
   • rouge_1_precision: 0.1128
   • rouge_1_recall: 0.7500
   • rouge_2: 0.0132
   • rouge_2_precision: 0.0076
   • rouge_2_recall: 0.0526
   • rouge_l: 0.0784
   • rouge_l_precision: 0.0451
   • rouge_l_recall: 0.3000

💡 Tips:
   • Higher BLEU/ROUGE scores indicate better similarity to reference commentaries
   • ROUGE-1

In [16]:
# Commentary Quality Evaluation Metrics
import re
from collections import Counter

def count_sentences(text):
    """Count sentences in text."""
    # Remove extra whitespace
    text = text.strip()
    if not text:
        return 0
    # Split by sentence endings
    sentences = re.split(r'[.!?]+', text)
    # Filter out empty strings
    sentences = [s.strip() for s in sentences if s.strip()]
    return len(sentences)

def count_words(text):
    """Count words in text."""
    words = re.findall(r'\b\w+\b', text)
    return len(words)

def analyze_readability(text):
    """Basic readability analysis."""
    sentences = count_sentences(text)
    words = count_words(text)
    chars = len(text)
    
    if sentences == 0:
        return {
            'avg_words_per_sentence': 0,
            'avg_chars_per_word': 0,
            'readability_level': 'N/A'
        }
    
    avg_words_per_sentence = words / sentences if sentences > 0 else 0
    avg_chars_per_word = chars / words if words > 0 else 0
    
    # Simple readability classification
    if avg_words_per_sentence < 10:
        level = 'Simple'
    elif avg_words_per_sentence < 20:
        level = 'Moderate'
    else:
        level = 'Complex'
    
    return {
        'avg_words_per_sentence': round(avg_words_per_sentence, 1),
        'avg_chars_per_word': round(avg_chars_per_word, 1),
        'readability_level': level
    }

def check_chess_terminology(text):
    """Check for chess-specific terminology."""
    chess_terms = [
        'pawn', 'knight', 'bishop', 'rook', 'queen', 'king',
        'castling', 'check', 'mate', 'checkmate',
        'fork', 'pin', 'skewer', 'discovered',
        'en passant', 'promotion',
        'opening', 'middlegame', 'endgame',
        'development', 'control', 'threat',
        'attack', 'defense', 'tempo'
    ]
    
    text_lower = text.lower()
    found_terms = [term for term in chess_terms if term in text_lower]
    
    return {
        'chess_terms_count': len(found_terms),
        'chess_terms_found': found_terms[:10],  # Show first 10
        'terminology_score': min(len(found_terms) / 10, 1.0)  # Normalized to 0-1
    }

def evaluate_commentary(commentary_text, move_info=None):
    """
    Comprehensive evaluation of commentary quality.
    
    Args:
        commentary_text: Generated commentary string
        move_info: Optional dict with move details
        
    Returns:
        Dictionary with evaluation metrics
    """
    if not commentary_text or not commentary_text.strip():
        return {
            'error': 'Empty commentary',
            'sentence_count': 0,
            'word_count': 0,
            'meets_minimum': False
        }
    
    # Basic metrics
    sentence_count = count_sentences(commentary_text)
    word_count = count_words(commentary_text)
    char_count = len(commentary_text)
    
    # Requirements check
    meets_minimum_sentences = sentence_count >= 2
    
    # Readability analysis
    readability = analyze_readability(commentary_text)
    
    # Chess terminology check
    chess_analysis = check_chess_terminology(commentary_text)
    
    # Structure analysis
    has_ending_punctuation = commentary_text.strip().endswith(('.', '!', '?'))
    
    # Quality indicators
    has_move_reference = bool(re.search(r'\d+\.?\s*\w+', commentary_text)) or 'move' in commentary_text.lower()
    has_evaluation_mention = bool(re.search(r'advantage|disadvantage|equal|position|evaluation', commentary_text, re.I))
    
    # Coherence indicators
    transition_words = ['however', 'therefore', 'indeed', 'furthermore', 'meanwhile', 'consequently', 'although', 'while']
    has_transitions = any(word in commentary_text.lower() for word in transition_words)
    
    # Overall quality score (0-1)
    quality_score = (
        (0.3 if meets_minimum_sentences else 0) +
        (0.2 if 50 <= word_count <= 200 else 0.1) +  # Ideal length
        (0.2 * chess_analysis['terminology_score']) +
        (0.1 if has_ending_punctuation else 0) +
        (0.1 if has_move_reference else 0) +
        (0.1 if has_evaluation_mention else 0)
    )
    
    metrics = {
        'sentence_count': sentence_count,
        'word_count': word_count,
        'char_count': char_count,
        'meets_minimum_sentences': meets_minimum_sentences,
        'readability': readability,
        'chess_terminology': chess_analysis,
        'structure': {
            'has_ending_punctuation': has_ending_punctuation,
            'has_move_reference': has_move_reference,
            'has_evaluation_mention': has_evaluation_mention,
            'has_transitions': has_transitions
        },
        'quality_score': round(quality_score, 2),
        'quality_rating': (
            'Excellent' if quality_score >= 0.8 else
            'Good' if quality_score >= 0.6 else
            'Fair' if quality_score >= 0.4 else
            'Needs Improvement'
        )
    }
    
    return metrics

def print_evaluation_report(evaluation_metrics, move_info=None):
    """Print a formatted evaluation report."""
    print("="*60)
    print("COMMENTARY QUALITY EVALUATION")
    print("="*60)
    
    if move_info:
        print(f"Move: {move_info.get('move_number', 'N/A')}. {move_info.get('move_san', 'N/A')}")
        print()
    
    # Basic metrics
    print("📊 Basic Metrics:")
    print(f"  • Sentences: {evaluation_metrics['sentence_count']} {'✅' if evaluation_metrics['meets_minimum_sentences'] else '❌'}")
    print(f"  • Words: {evaluation_metrics['word_count']}")
    print(f"  • Characters: {evaluation_metrics['char_count']}")
    print()
    
    # Requirements
    print("✅ Requirements Check:")
    print(f"  • Meets minimum 2 sentences: {'✅ YES' if evaluation_metrics['meets_minimum_sentences'] else '❌ NO'}")
    print()
    
    # Readability
    print("📖 Readability:")
    read = evaluation_metrics['readability']
    print(f"  • Avg words/sentence: {read['avg_words_per_sentence']}")
    print(f"  • Readability level: {read['readability_level']}")
    print()
    
    # Chess terminology
    print("♟️  Chess Terminology:")
    chess = evaluation_metrics['chess_terminology']
    print(f"  • Chess terms found: {chess['chess_terms_count']}")
    if chess['chess_terms_found']:
        print(f"  • Terms: {', '.join(chess['chess_terms_found'][:5])}")
        if len(chess['chess_terms_found']) > 5:
            print(f"    ... and {len(chess['chess_terms_found']) - 5} more")
    print()
    
    # Structure
    print("🏗️  Structure Quality:")
    struct = evaluation_metrics['structure']
    print(f"  • Proper ending: {'✅' if struct['has_ending_punctuation'] else '❌'}")
    print(f"  • Move reference: {'✅' if struct['has_move_reference'] else '❌'}")
    print(f"  • Evaluation mention: {'✅' if struct['has_evaluation_mention'] else '❌'}")
    print(f"  • Transition words: {'✅' if struct['has_transitions'] else '❌'}")
    print()
    
    # Overall quality
    print("⭐ Overall Quality:")
    print(f"  • Quality Score: {evaluation_metrics['quality_score']:.2f}/1.00")
    print(f"  • Rating: {evaluation_metrics['quality_rating']}")
    print("="*60)

# Test with the generated commentary
if 'result' in locals():
    print("Evaluating Commentary Quality:")
    print()
    
    # Handle all possible result formats
    moves_list = []
    
    # Import CommentaryResult to check type
    try:
        from deepchessiq_nlp.models.schemas import CommentaryResult, GameAnalysis
    except ImportError:
        try:
            from models.schemas import CommentaryResult, GameAnalysis
        except ImportError:
            CommentaryResult = None
            GameAnalysis = None
    
    # Check if result is a single CommentaryResult object
    if CommentaryResult and isinstance(result, CommentaryResult):
        moves_list.append({
            'move_number': result.move_analysis.move_number,
            'move_san': result.move_analysis.move_san,
            'commentary': result.commentary,
            'evaluation': {
                'score': result.move_analysis.evaluation.score,
                'best_move': result.move_analysis.evaluation.best_move
            }
        })
    # Check if result is a GameAnalysis object (from pipeline.process_game)
    elif GameAnalysis and isinstance(result, GameAnalysis):
        # Convert CommentaryResult objects to dict format
        for c_result in result.moves:
            moves_list.append({
                'move_number': c_result.move_analysis.move_number,
                'move_san': c_result.move_analysis.move_san,
                'commentary': c_result.commentary,
                'evaluation': {
                    'score': c_result.move_analysis.evaluation.score,
                    'best_move': c_result.move_analysis.evaluation.best_move
                }
            })
    # Check if result is a dict (from generate_commentary function)
    elif isinstance(result, dict):
        if 'game_analysis' in result and 'moves' in result['game_analysis']:
            moves_list = result['game_analysis']['moves']
        elif 'commentary' in result:
            # Single move result
            moves_list = [result]
    # Fallback: check for moves attribute
    elif hasattr(result, 'moves'):
        # GameAnalysis-like object
        for c_result in result.moves:
            moves_list.append({
                'move_number': c_result.move_analysis.move_number,
                'move_san': c_result.move_analysis.move_san,
                'commentary': c_result.commentary,
                'evaluation': {
                    'score': c_result.move_analysis.evaluation.score,
                    'best_move': c_result.move_analysis.evaluation.best_move
                }
            })
    # Fallback: single CommentaryResult-like object
    elif hasattr(result, 'commentary') and hasattr(result, 'move_analysis'):
        moves_list.append({
            'move_number': result.move_analysis.move_number,
            'move_san': result.move_analysis.move_san,
            'commentary': result.commentary,
            'evaluation': {
                'score': result.move_analysis.evaluation.score,
                'best_move': result.move_analysis.evaluation.best_move
            }
        })
    
    if not moves_list:
        print("⚠ No moves found in result. Check the result structure.")
        print(f"Result type: {type(result)}")
        if isinstance(result, dict):
            print(f"Result keys: {list(result.keys())}")
        elif hasattr(result, '__dict__'):
            print(f"Result attributes: {list(result.__dict__.keys())}")
    else:
        # Evaluate all moves
        all_metrics = []
        for move in moves_list:
            # Handle both dict and object access
            if isinstance(move, dict):
                move_num = move.get('move_number', 'N/A')
                move_san = move.get('move_san', 'N/A')
                commentary = move.get('commentary', '')
            else:
                # Handle CommentaryResult object
                move_num = move.move_analysis.move_number
                move_san = move.move_analysis.move_san
                commentary = move.commentary
            
            if not commentary:
                print(f"⚠ Move {move_num}. {move_san}: No commentary found, skipping...")
                continue
            
            metrics = evaluate_commentary(commentary, {'move_number': move_num, 'move_san': move_san})
            metrics['move_number'] = move_num
            metrics['move_san'] = move_san
            all_metrics.append(metrics)
            
            print_evaluation_report(metrics, {'move_number': move_num, 'move_san': move_san})
            print()
        
        # Summary statistics
        if all_metrics:
            print("="*60)
            print("SUMMARY STATISTICS")
            print("="*60)
            print(f"Total moves evaluated: {len(all_metrics)}")
            print(f"Average sentences per move: {sum(m['sentence_count'] for m in all_metrics) / len(all_metrics):.1f}")
            print(f"Average words per move: {sum(m['word_count'] for m in all_metrics) / len(all_metrics):.0f}")
            print(f"Average quality score: {sum(m['quality_score'] for m in all_metrics) / len(all_metrics):.2f}")
            print(f"Moves meeting minimum (2+ sentences): {sum(1 for m in all_metrics if m['meets_minimum_sentences'])}/{len(all_metrics)}")
            print("="*60)
        else:
            print("⚠ No valid commentary found to evaluate.")
else:
    print("⚠ No results found. Run the game processing cell first.")


Evaluating Commentary Quality:

COMMENTARY QUALITY EVALUATION
Move: 3. Nf3

📊 Basic Metrics:
  • Sentences: 5 ✅
  • Words: 106
  • Characters: 726

✅ Requirements Check:
  • Meets minimum 2 sentences: ✅ YES

📖 Readability:
  • Avg words/sentence: 21.2
  • Readability level: Complex

♟️  Chess Terminology:
  • Chess terms found: 8
  • Terms: knight, king, castling, check, fork
    ... and 3 more

🏗️  Structure Quality:
  • Proper ending: ✅
  • Move reference: ✅
  • Evaluation mention: ✅
  • Transition words: ✅

⭐ Overall Quality:
  • Quality Score: 0.96/1.00
  • Rating: Excellent

SUMMARY STATISTICS
Total moves evaluated: 1
Average sentences per move: 5.0
Average words per move: 106
Average quality score: 0.96
Moves meeting minimum (2+ sentences): 1/1


## 10. Performance Evaluation

This section evaluates the **performance metrics** of the commentary generation service:

- ⏱️ **Latency Metrics**: Generation time per move, average response time
- 🚀 **Throughput**: Commentaries per minute/hour
- 💾 **Resource Usage**: GPU memory, CPU utilization, RAM usage
- 📊 **Batch Processing**: Performance comparison (single vs batch)
- 📈 **Scalability**: Time complexity analysis


In [17]:
# Process a complete game
print("Processing Complete Game:")
print("=" * 60)
print("⚠️  Processing first 5 moves only (for demo)")

try:
    if 'pipeline' in locals():
        result = generate_commentary(
            pgn_path="data/sample_game.pgn",
            max_moves=5  # Limit for demo
        )
        
        print(f"\n✓ Processed {result['game_analysis']['total_moves']} moves")
        print(f"Game result: {result['game_analysis']['result']}")
        
        print("\nGame metadata:")
        for key, value in result['game_analysis']['metadata'].items():
            print(f"  {key}: {value}")
    else:
        print("✗ Pipeline not initialized. Run the LLM initialization cells first.")
        
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

print("=" * 60)


Processing Complete Game:
⚠️  Processing first 5 moves only (for demo)
2025-11-28 19:01:10,234 - deepchessiq_nlp - INFO - _detect_device:39 - Using GPU: NVIDIA GeForce RTX 4070 Laptop GPU
2025-11-28 19:01:10,236 - deepchessiq_nlp - INFO - __init__:78 - ✓ CommentaryPipeline initialized
2025-11-28 19:01:10,246 - deepchessiq_nlp - INFO - parse_pgn:35 - Successfully parsed PGN: Sample Game
2025-11-28 19:01:10,302 - deepchessiq_nlp - INFO - process_game:114 - Processing 5 moves
2025-11-28 19:01:10,310 - deepchessiq_nlp - INFO - process_game:121 - Move 1: e4
2025-11-28 19:01:10,324 - deepchessiq_nlp - INFO - load_model:52 - Loading model: microsoft/Phi-3-mini-4k-instruct
2025-11-28 19:01:10,329 - deepchessiq_nlp - INFO - load_model:53 - This may take 2-5 minutes and use significant memory...

📥 DOWNLOAD PROGRESS
Model: microsoft/Phi-3-mini-4k-instruct
First-time download may take 5-15 minutes (normal)
Progress bars will appear below as files download...

2025-11-28 19:01:10,331 - deepchessiq

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████| 2/2 [00:16<00:00,  8.22s/it]



✓ MODEL WEIGHTS DOWNLOADED AND LOADED

2025-11-28 19:01:28,798 - deepchessiq_nlp - INFO - load_model:189 - Successfully loaded model: microsoft/Phi-3-mini-4k-instruct
2025-11-28 19:01:55,791 - deepchessiq_nlp - INFO - generate_commentary:87 - Generated commentary with 4 sentences
2025-11-28 19:01:55,791 - deepchessiq_nlp - INFO - process_game:121 - Move 2: e5
2025-11-28 19:02:17,516 - deepchessiq_nlp - INFO - generate_commentary:87 - Generated commentary with 4 sentences
2025-11-28 19:02:17,516 - deepchessiq_nlp - INFO - process_game:121 - Move 3: Nf3
2025-11-28 19:02:51,486 - deepchessiq_nlp - INFO - generate_commentary:87 - Generated commentary with 5 sentences
2025-11-28 19:02:51,486 - deepchessiq_nlp - INFO - process_game:121 - Move 4: Nc6
2025-11-28 19:03:17,893 - deepchessiq_nlp - INFO - generate_commentary:87 - Generated commentary with 4 sentences
2025-11-28 19:03:17,893 - deepchessiq_nlp - INFO - process_game:121 - Move 5: Bb5
2025-11-28 19:03:52,277 - deepchessiq_nlp - INFO 

In [18]:
# Display commentary for each move
if 'result' in locals():
    print("Generated Commentary for Each Move:")
    print("=" * 60)
    
    for move in result['game_analysis']['moves']:
        print(f"\n{'='*60}")
        print(f"Move {move['move_number']}: {move['move_san']}")
        print(f"{'='*60}")
        print(f"Evaluation: {move['evaluation']['score']:+.2f} pawns")
        print(f"Best move: {move['evaluation']['best_move']}")
        
        if move['is_turning_point']:
            print("⚠️  TURNING POINT")
        
        if move['tactical_motifs']:
            motif_names = [m['name'] for m in move['tactical_motifs']]
            print(f"Tactical motifs: {', '.join(motif_names)}")
        
        print(f"\nCommentary:")
        print(f"{move['commentary']}")
        
        print(f"\nMetadata:")
        print(f"  Model: {move['metadata'].get('model', 'N/A')}")
        print(f"  Hallucination check: {move['metadata'].get('hallucination_check', {}).get('passed', 'N/A')}")
    
    print("\n" + "=" * 60)
else:
    print("No results to display. Run the game processing cell first.")


Generated Commentary for Each Move:

Move 1: e4
Evaluation: -2.00 pawns
Best move: Nh6
Tactical motifs: fork, castling_opportunity, castling_opportunity, castling_opportunity, castling_opportunity, undeveloped_piece, undeveloped_piece, undeveloped_piece, undeveloped_piece

Commentary:
The opening with 1. e4 is highly aggressive, but Black'ieves an immediate developmental leap after responding appropriately; however, given White’s lack of coordination between pieces post-move, it seems like they may have missed out on significant opportunities such as creating a fork or capitalizing early castling options which could lead to substantial positional advantages down the line. A more prudent approach might involve moves that not only contest central control but also develop minor pieces towards active squares without sacrificing material—perhaps leading toward alternatives suggested by top engines favoring positions where knights can jump into play via h5 instead of being developed passivel

## 9. Save Results and Cleanup


In [19]:
# Save results to JSON
if 'result' in locals():
    output_path = "commentary_output.json"
    
    with open(output_path, 'w') as f:
        json.dump(result, f, indent=2)
    
    print(f"✓ Results saved to: {output_path}")
    print(f"   File size: {Path(output_path).stat().st_size / 1024:.2f} KB")
else:
    print("No results to save. Run the game processing cell first.")

# Cleanup resources
if 'pipeline' in locals():
    print("\nCleaning up resources...")
    pipeline.cleanup()
    print("✓ Cleanup complete")
else:
    print("No pipeline to clean up.")


✓ Results saved to: commentary_output.json
   File size: 13.49 KB

Cleaning up resources...
2025-11-28 19:05:15,421 - deepchessiq_nlp - INFO - unload_model:198 - Unloading LLM from GPU...
2025-11-28 19:05:15,704 - deepchessiq_nlp - INFO - unload_model:216 - ✓ Model completely unloaded
✓ Cleanup complete


## 10. Integration Examples

Examples showing how to integrate this module with RL agents and backend APIs.


In [20]:
# Example: RL Agent Integration
def rl_agent_commentary_example():
    """
    Example function showing how RL team can integrate commentary.
    """
    # RL agent has made a move
    current_position = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"
    last_move = "e5"
    move_history = ["e4"]
    
    # Generate commentary
    result = generate_commentary(
        position_fen=current_position,
        move_san=last_move
    )
    
    # Return structured data for UI
    return {
        "move": last_move,
        "commentary": result["commentary"],
        "evaluation": result["evaluation"],
        "best_move": result["evaluation"]["best_move"]
    }

print("RL Agent Integration Example:")
print("=" * 60)
print("Function definition created.")
print("Call rl_agent_commentary_example() to generate commentary for RL agent moves.")
print("=" * 60)


RL Agent Integration Example:
Function definition created.
Call rl_agent_commentary_example() to generate commentary for RL agent moves.


## Summary

This notebook demonstrates all features of the DeepChessIQ NLP module:

✅ **Stockfish Integration** - Engine evaluation and best move analysis  
✅ **PGN Parsing** - Extract games and positions from PGN files  
✅ **Feature Extraction** - Material, development, center control analysis  
✅ **Tactical Motifs** - Detect pins, forks, skewers, and other tactics  
✅ **Turning Points** - Identify critical moments in games  
✅ **LLM Commentary** - Generate professional chess commentary  
✅ **Hallucination Filtering** - Ensure commentary accuracy  
✅ **Complete Pipeline** - Process full games with commentary  

**Next Steps:**
- Use this module in your RL agent integration
- Connect to your backend API
- Fine-tune the LLM model on chess commentary datasets
- Customize commentary style and length
